In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import os
import torch
import numpy as np

from constants import YOLO_SAVED_MODELS, EXPERIMENT_RESULTS

/home/blukash/anaconda3/envs/yolo_5/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = torch.hub.load('ultralytics/yolov5', 'custom', path=YOLO_SAVED_MODELS.BEST_YOLO_MODEL_PATH)

Using cache found in /home/blukash/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2022-5-9 torch 1.11.0+cu102 CUDA:0 (NVIDIA GeForce 940MX, 2004MiB)

Fusing layers... 
Model summary: 280 layers, 12342868 parameters, 0 gradients, 16.2 GFLOPs
Adding AutoShape... 


In [4]:
digits_paths = [f'../datasets/ruler_digit/images/val/{img}' for img in os.listdir('../datasets/ruler_digit/images/val/')]

In [5]:
results = [model(img) for img in digits_paths]

In [6]:
import cv2

img = cv2.imread(digits_paths[1])

for idx, detection in results[1].pandas().xyxy[0].iterrows():
    xmin = round(detection['xmin'])
    xmax = round(detection['xmax'])
    ymin = round(detection['ymin'])
    ymax = round(detection['ymax'])
    cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (0, 255, 0), 3)
    cv2.putText(img, str(detection['name']), (xmin, ymin), fontFace=cv2.FONT_HERSHEY_TRIPLEX, fontScale=3, color=(0, 255, 0), thickness=3)

cv2.namedWindow('image', cv2.WINDOW_NORMAL)
cv2.imshow('image', img)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [7]:
from ray_casting import Point, Edge, Rect

In [8]:
# 0 - xmin, 1 - ymin, 2 - xmax, 3 - ymax, 4 - confidence, 5 - class (name = class + 1)

In [9]:
def get_prediction_poly(prediction):
        xmin = prediction[0]
        xmax = prediction[2]
        ymin = prediction[1]
        ymax = prediction[3]
        
        prediction_name = prediction[5] + 1
        return Rect(name=prediction_name, edges=(
            Edge(a=Point(x=xmin, y=ymin), b=Point(x=xmax, y=ymin)),
            Edge(a=Point(x=xmax, y=ymin), b=Point(x=xmax, y=ymax)),
            Edge(a=Point(x=xmax, y=ymax), b=Point(x=xmin, y=ymax)),
            Edge(a=Point(x=xmin, y=ymax), b=Point(x=xmin, y=ymin))
        ))

In [10]:
import math

In [11]:
def sort_predictions(predictions, by=5):
    return predictions[predictions[:,by].argsort()]

In [12]:
def detect_candidate_points(predictions):
    sorted_predictions = sort_predictions(predictions.xyxy[0].cpu().numpy())

    for idx, prediction in enumerate(sorted_predictions):
        test_p1 = get_prediction_poly(prediction)
        test_center_p1 = test_p1.center()

        #delete current test prediction from the rest
        rest_predictions = np.delete(sorted_predictions, idx, 0)

        while len(rest_predictions):
            test_prediction_2 = rest_predictions[-1]

            #delete current test 2 prediction (last elem) from the rest
            rest_predictions = np.delete(rest_predictions, -1, 0)

            test_p2 = get_prediction_poly(test_prediction_2)

            test_center_p2 = test_p2.center()

            test_edge = Edge(test_center_p1, test_center_p2)
            print(test_edge)

            intersected_digits = []
            for test_prediction_n in rest_predictions:
                digit = get_prediction_poly(test_prediction_n)
                print(digit)
                is_intersecting_digit = digit.ray_goes_through(test_edge)

                test_prediction_n_name = test_prediction_n[5] + 1
                print(f"Digit {test_prediction_n_name} is {'intersected' if is_intersecting_digit else 'NOT intersected'}")

                intersected_digits = [*intersected_digits, digit] if is_intersecting_digit else intersected_digits

                # if we have at least 3 points on one line (test one + intersected)
                digits_on_one_line = [test_p1, *intersected_digits, test_p2]
                if len(digits_on_one_line) > math.floor(len(sorted_predictions) / 2):
                    digit_center_points = [digit.center() for digit in digits_on_one_line]

                    return digit_center_points, digits_on_one_line

In [14]:
import cv2

img = cv2.imread(digits_paths[1])

candidates = []
for idx, detection in results[1].pandas().xyxy[0].iterrows():
    xmin = round(detection['xmin'])
    xmax = round(detection['xmax'])
    ymin = round(detection['ymin'])
    ymax = round(detection['ymax'])
    cv2.putText(img, str(detection['name']), (xmin, ymin - 15), fontFace=cv2.FONT_HERSHEY_TRIPLEX, fontScale=3, color=(0, 255, 0), thickness=3)
    if detection['name'] == '1' or detection['name'] == '9':
        cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (0, 0, 255), 3)
        candidates.append(detection)
        continue

    cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (0, 255, 0), 3)


one = get_prediction_poly(candidates[0]).center()
nine = get_prediction_poly(candidates[1]).center()

cv2.line(img, (round(one.x), round(one.y)), (round(nine.x), round(nine.y)), (0, 165, 255), 3)
cv2.namedWindow('image', cv2.WINDOW_NORMAL)
cv2.imshow('image', img)

cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.imwrite('../paper/1-9-line.jpg', img)

True

In [22]:
def get_mean_segments_len(edges):
    return np.mean([len(edge) for edge in edges])

In [20]:
def create_candidate_edges(candidate_points, digits):
    edges = []
    segments = []
    for idx, point in enumerate(candidate_points):
        is_last_digit = idx == len(candidate_points) - 1

        if is_last_digit:
            return edges, segments

        tast_edge = Edge(point, candidate_points[idx + 1])

        has_simmilar_test_edge_length = (len(edges) == 0) or (len(tast_edge) == get_mean_segments_len(edges))
        next_digit = int(digits[idx + 1].name)
        current_digit = int(digits[idx].name)

        if next_digit == current_digit + 1 and has_simmilar_test_edge_length:
            edges.append(tast_edge)
            segments.append((digits[idx].name, digits[idx + 1].name))

    return edges, segments

In [17]:
candidate_points, digits = detect_candidate_points(results[6])

(x: 943.9695129394531, y: 1685.9981079101562), (x: 3555.3167724609375, y: 1342.8243408203125)
2.0, (x: 1234.758544921875, y: 1607.3079833984375), (x: 1295.9576416015625, y: 1607.3079833984375) (x: 1295.9576416015625, y: 1607.3079833984375), (x: 1295.9576416015625, y: 1684.041748046875) (x: 1295.9576416015625, y: 1684.041748046875), (x: 1234.758544921875, y: 1684.041748046875) (x: 1234.758544921875, y: 1684.041748046875), (x: 1234.758544921875, y: 1607.3079833984375)
Digit 2.0 is intersected
3.0, (x: 1574.025634765625, y: 1567.4365234375), (x: 1628.3958740234375, y: 1567.4365234375) (x: 1628.3958740234375, y: 1567.4365234375), (x: 1628.3958740234375, y: 1635.68994140625) (x: 1628.3958740234375, y: 1635.68994140625), (x: 1574.025634765625, y: 1635.68994140625) (x: 1574.025634765625, y: 1635.68994140625), (x: 1574.025634765625, y: 1567.4365234375)
Digit 3.0 is intersected
4.0, (x: 1900.3455810546875, y: 1528.479736328125), (x: 1956.178466796875, y: 1528.479736328125) (x: 1956.178466796875

In [23]:
candidate_edges, candidate_segments = create_candidate_edges(candidate_points, digits)

In [24]:
for edge in candidate_segments:
    print(edge)

(1.0, 2.0)


In [25]:
mean_len = get_mean_segments_len(candidate_edges)

In [26]:
mean_len

324.0

In [ ]:
import random

In [ ]:
import cv2

img = cv2.imread(digits_paths[1])

for idx, detection in results[1].pandas().xyxy[0].iterrows():
    xmin = round(detection['xmin'])
    xmax = round(detection['xmax'])
    ymin = round(detection['ymin'])
    ymax = round(detection['ymax'])
    cv2.putText(img, str(detection['name']), (xmin, ymin - 15), fontFace=cv2.FONT_HERSHEY_TRIPLEX, fontScale=3, color=(0, 255, 0), thickness=3)

    if detection['name'] == '1' or detection['name'] == '9':
        cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (0, 0, 255), 3)
        candidates.append(detection)
        continue

    cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (0, 255, 0), 3)

random_edge_idx = random.randint(0, len(candidate_edges) - 1)
edge = candidate_edges[random_edge_idx]
color = list(np.random.random(size=3) * 256)
cv2.putText(img, str(mean_len), (round(edge.a.x + 70), round(edge.a.y) - 50), fontFace=cv2.FONT_HERSHEY_TRIPLEX, fontScale=3, color=color, thickness=3)
cv2.line(img, (round(edge.a.x), round(edge.a.y)), (round(edge.b.x), round(edge.b.y)), color, 6)

cv2.namedWindow('image', cv2.WINDOW_NORMAL)
cv2.imshow('image', img)

cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.imwrite('../paper/scale_detection_ex_3.jpg', img)

True